# Associational Biases — Colab Runner

**Workflow:** Edit locally in Cursor → `git push` → run this notebook on Colab.

**Runtime:** `Runtime → Change runtime type → A100 GPU`

---

## 1. Session Setup

Run cells **1a → 1e** every time you open this notebook. They're idempotent and fast (~2 min).

In [ ]:
#@title 1a. GPU check + mount Google Drive
import torch, os, pathlib

assert torch.cuda.is_available(), "No GPU — set Runtime → A100"
print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT  = pathlib.Path("/content/drive/MyDrive/associational_biases")
CACHE_DIR   = DRIVE_ROOT / "hf_cache"
DATASET_DIR = DRIVE_ROOT / "datasets"
RESULTS_DIR = DRIVE_ROOT / "results"

for d in [DRIVE_ROOT, CACHE_DIR, DATASET_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive root: {DRIVE_ROOT}")

In [ ]:
#@title 1b. Pull your repo (clones on first run, pulls on subsequent runs)

# ──── EDIT THIS: your GitHub repo URL ────
GITHUB_REPO = "https://github.com/alpozaydin/Associational_Biases.git"
BRANCH      = "main"
# ─────────────────────────────────────────

REPO_DIR     = pathlib.Path("/content/repo")
PIPELINE_DIR = REPO_DIR / "inter-model communication pipeline"
EXPLAIN_DIR  = REPO_DIR / "explainability_pipeline"

if (REPO_DIR / ".git").exists():
    print("Repo exists — pulling latest changes…")
    !cd "{REPO_DIR}" && git fetch origin && git reset --hard origin/{BRANCH}
else:
    print("Cloning repo…")
    !git clone -b {BRANCH} "{GITHUB_REPO}" "{REPO_DIR}"

!cd "{REPO_DIR}" && git log --oneline -3
print(f"\nPipeline dir: {PIPELINE_DIR}")

In [ ]:
#@title 1c. Install dependencies
!pip install -q \
    torch torchvision \
    "diffusers>=0.32" "transformers>=4.48" accelerate \
    safetensors "huggingface-hub[cli]" \
    sentence-transformers scipy \
    pandas pyyaml pydantic python-dotenv \
    protobuf sentencepiece tqdm aiohttp \
    matplotlib adjustText "krippendorff==0.6.0" \
    gdown statsmodels \
    pytorch-grad-cam retina-face mediapipe opencv-python-headless

In [ ]:
#@title 1d. Set environment variables + HF login
import os

# ──── EDIT THIS ────
HF_TOKEN = "hf_PASTE_YOUR_TOKEN_HERE"   # https://huggingface.co/settings/tokens
# ───────────────────

os.environ["HF_TOKEN"]           = HF_TOKEN
os.environ["HF_HOME"]            = str(CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR)
os.environ["HF_HUB_CACHE"]       = str(CACHE_DIR)

!huggingface-cli login --token "{HF_TOKEN}" --add-to-git-credential 2>/dev/null
print(f"Cache dir: {CACHE_DIR}  (persists on Drive)")

In [ ]:
#@title 1e. Set working directory + create dataset symlinks
import sys

%cd "{PIPELINE_DIR}"

if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

# Symlink datasets into the repo so the code finds them at expected paths
def ensure_symlink(target, link_name):
    link = pathlib.Path(link_name)
    if link.is_symlink() or link.exists():
        return
    target = pathlib.Path(target)
    if target.exists():
        link.symlink_to(target)
        print(f"  linked: {link_name} → {target}")
    else:
        print(f"  ⚠ upload dataset first: {target}")

ensure_symlink(DATASET_DIR / "RAF_DB",     PIPELINE_DIR / "RAF_DB")
ensure_symlink(DATASET_DIR / "phase_imgs", PIPELINE_DIR / "phase_imgs")

from utils.initialisation import DEVICE
print(f"\nDevice: {DEVICE}  |  Setup complete.")

---

## 2. Datasets

Upload to Google Drive **once**. Expected structure:

```
Google Drive/
└── associational_biases/
    ├── datasets/
    │   ├── RAF_DB/
    │   │   └── DATASET/
    │   │       ├── test/1/ … /7/     ← emotion folders with .jpg images
    │   │       └── train/1/ … /7/
    │   └── phase_imgs/
    │       ├── train/*.png
    │       └── val/*.png
    ├── hf_cache/          ← auto-managed (model weights)
    └── results/           ← auto-managed (pipeline outputs)
```

In [ ]:
#@title 2a. Verify dataset presence
for name, expected in [
    ("RAF_DB test",  DATASET_DIR / "RAF_DB" / "DATASET" / "test"),
    ("RAF_DB train", DATASET_DIR / "RAF_DB" / "DATASET" / "train"),
    ("PHASE train",  DATASET_DIR / "phase_imgs" / "train"),
    ("PHASE val",    DATASET_DIR / "phase_imgs" / "val"),
]:
    exists = expected.exists()
    count  = len(list(expected.iterdir())) if exists else 0
    status = f"✓ {count} items" if exists else "✗ MISSING"
    print(f"  {name:15s}  {status}")

---

## 3. Run: Inter-Model Communication Pipeline

In [ ]:
#@title 3a. Load models into GPU (once per session, ~3 min first time, <1 min cached)
from utils.initialisation import init_stable_diffusion, init_llava

print("Loading Stable Diffusion 3.5 Large…")
pipe = init_stable_diffusion()

print("Loading LLaVA v1.6 Mistral 7B…")
processor, llava_model = init_llava()

torch.cuda.empty_cache()
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("Models loaded.")

In [ ]:
#@title 3b. RAF-DB test split: describe → generate → describe
%cd "{PIPELINE_DIR}"
!python main.py --raf --raf_split test -o "{RESULTS_DIR}/collected_raf"

In [ ]:
#@title 3c. RAF-DB train split
%cd "{PIPELINE_DIR}"
!python main.py --raf --raf_split train -o "{RESULTS_DIR}/collected_raf_train"

In [ ]:
#@title 3d. PHASE activities
%cd "{PIPELINE_DIR}"
!python phase.py -o "{RESULTS_DIR}/phase_acts" -c activities

In [ ]:
#@title 3e. PHASE emotions
%cd "{PIPELINE_DIR}"
!python phase.py -o "{RESULTS_DIR}/phase_emos" -c emotions

---

## 4. Run: Evaluation & Annotation

In [ ]:
#@title 4a. Annotate RAF generated images (demographic labels via LLaVA)
%cd "{PIPELINE_DIR}"
!python -m eval.raf_img_eval \
    -j "{RESULTS_DIR}/collected_raf/results.json" \
    -o "{RESULTS_DIR}/collected_raf/annotated_results.json" \
    -p raf

In [ ]:
#@title 4b. Annotate PHASE generated images
%cd "{PIPELINE_DIR}"
!python -m eval.phase_img_eval

In [ ]:
#@title 4c. Chi-squared statistical tests
%cd "{PIPELINE_DIR}"
!python -m eval.chi

---

## 5. Run: Explainability Pipeline

Sequential: image selection → face detection → hair segmentation → filtering → GradCAM.

In [ ]:
#@title 5a. Random image selection (RAF example — change emotion as needed)
import sys
sys.path.insert(0, str(EXPLAIN_DIR))

from random_image_selection.raf_image_selection import get_raf_images

EMOTION = "anger"  # change to: surprise, fear, disgust, happiness, sadness, anger, neutral

get_raf_images(
    emotion=EMOTION,
    root_dir=str(DATASET_DIR / "RAF_DB"),
    subset="all",
    num_to_copy="max"
)

In [ ]:
#@title 5b. RetinaFace — detect faces
from retina_face.get_face_bboxes import get_retina_face_outputs

INPUT_IMAGES = str(DATASET_DIR / "RAF_DB" / f"raf_{EMOTION}_images")
FACE_OUT     = str(RESULTS_DIR / "explainability" / EMOTION / "faces")
FACE_JSON    = f"{FACE_OUT}/face_detections.json"

get_retina_face_outputs(
    input_dir=INPUT_IMAGES,
    output_dir=FACE_OUT,
    output_json=FACE_JSON,
    save_images=True
)

In [ ]:
#@title 5c. Hair segmentation
from hair_segmentation.get_hair_segmentation import get_hair_seg_outputs_raf

HAIR_OUT  = str(RESULTS_DIR / "explainability" / EMOTION / "hair_masks")
HAIR_JSON = f"{HAIR_OUT}/hair_segmentation.json"

# Download MediaPipe hair segmenter model if not present
!wget -q -nc -P "{EXPLAIN_DIR}/hair_segmentation" \
    "https://storage.googleapis.com/mediapipe-models/image_segmenter/hair_segmenter/float32/latest/hair_segmenter.tflite"

get_hair_seg_outputs_raf(
    input_dir=INPUT_IMAGES,
    output_dir=HAIR_OUT,
    retinaface_json=FACE_JSON,
    output_json=HAIR_JSON,
    hair_model=str(EXPLAIN_DIR / "hair_segmentation" / "hair_segmenter.tflite")
)

In [ ]:
#@title 5d. Hair/face filtering
from hair_face_filtering.raf_filtering import get_filtered_entries

FILTER_OUT = str(RESULTS_DIR / "explainability" / EMOTION)

get_filtered_entries(
    facial_annotation_file=FACE_JSON,
    hair_segmentation_file=HAIR_JSON,
    output_dir=FILTER_OUT,
    category=EMOTION
)

For **GradCAM** (step 5e), see `explainability_pipeline/gradcam_pipeline/run_gradcam_pipeline.ipynb`.

---

## 6. Smoke Test

Quick sanity check — no datasets needed.

In [ ]:
#@title Generate one image and describe it
%cd "{PIPELINE_DIR}"
from utils.initialisation import init_stable_diffusion, init_llava
from utils.description import describe_image_llava
import torch

pipe = init_stable_diffusion()
processor, model = init_llava()

prompt = "a photo of a person showing surprise in an office"
print(f"Prompt: {prompt}")

image = pipe(prompt, num_inference_steps=28, width=768, height=768).images[0]
display(image)

desc = describe_image_llava(processor, model, image)
print(f"\nLLaVA description:\n{desc}")